# 02 · NumPy 基础与 Shape 思维

> **学习目标**：掌握后续 PyTorch / RAG / 训练里**最常翻车**的三件事 —— shape、broadcasting、矩阵乘法。
>
> **预备**：会基本 Python；中学线代（知道矩阵乘法定义）。
>
> **为什么重要**：90% 的深度学习 bug 是 shape 不对。先在 NumPy 里把 shape 直觉建立起来，进 PyTorch 就只剩 GPU 那点事。

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
print('numpy', np.__version__)

## 1. 创建数组 — 4 种最常用方式

**核心字段**：`shape`（每维有多大）、`dtype`（每个元素什么类型）、`ndim`（几维）。看一个数组先看这三个。

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])               # 从 list
b = np.zeros((2, 3))                                # 全 0
c = np.ones((2, 3), dtype=np.float32)               # 全 1，指定 dtype
d = np.arange(12).reshape(3, 4)                     # 连续整数 + reshape
e = np.random.randn(2, 3)                           # 标准正态

for name, arr in [('a', a), ('b', b), ('c', c), ('d', d), ('e', e)]:
    print(f'{name}: shape={arr.shape} dtype={arr.dtype} ndim={arr.ndim}')
    print(arr, '\n')

## 2. 索引与切片 — 永远先想 shape

**口诀**：每个轴一个切片，用逗号分开。负数从尾巴数。`:` 是「全要」。

In [ ]:
x = np.arange(24).reshape(2, 3, 4)
print('shape:', x.shape)
print(x, '\n')

print('x[0]      shape:', x[0].shape, '<- 取第 0 个 2D matrix，丢一维')
print('x[0, 1]   shape:', x[0, 1].shape, '<- 取一行')
print('x[0, 1, 2]:', x[0, 1, 2], '<- 取一个元素')
print('x[:, 0]   shape:', x[:, 0].shape, '<- 每个 batch 的第 0 行')
print('x[..., -1] shape:', x[..., -1].shape, '<- 每个 (B, L) 的最后一维')

In [ ]:
# Boolean mask —— 后面做注意力 mask、过滤负样本必用
scores = np.array([0.1, 0.9, 0.3, 0.7, 0.5])
mask = scores > 0.4
print('mask:', mask)
print('选出:', scores[mask])

# 索引数组 (fancy indexing)
topk = np.argsort(scores)[-3:][::-1]   # top-3 索引（降序）
print('top-3 idx:', topk, 'top-3 score:', scores[topk])

## 3. Broadcasting — 最容易出 bug 的特性

**规则**：两个数组运算时，从**尾巴对齐**比较每个维度。要么相等、要么其中一个是 1，否则报错。

```
  (3, 4)
+    (4,)    -> 后者补成 (1, 4)，再 broadcast 成 (3, 4)，OK
= (3, 4)

  (3, 4)
+ (3,)       -> 尾巴对不齐 (4 vs 3)，报错
```

In [ ]:
X = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])  # (3, 4)

row_mean = X.mean(axis=1, keepdims=True)   # (3, 1)
col_mean = X.mean(axis=0, keepdims=True)   # (1, 4)

print('row_mean shape:', row_mean.shape, '\n', row_mean)
print('col_mean shape:', col_mean.shape, '\n', col_mean)

# 行归一化：每行减自己的均值 —— 这就是 LayerNorm 的中心化部分
X_centered = X - row_mean
print('\nX - row_mean:\n', X_centered)
print('每行和应该 ≈ 0:', X_centered.sum(axis=1))

In [ ]:
# 一个 broadcasting 翻车案例
try:
    a = np.zeros((3, 4))
    b = np.zeros((3,))             # ⚠ shape (3,) 会和最后一维 4 对齐 → 报错
    a + b
except ValueError as e:
    print('翻车:', e)

# 修法：显式 reshape 表明意图
b2 = b.reshape(3, 1)               # (3, 1) 和 (3, 4) 后面对齐 (4) vs (1) → 广播
print('OK:', (a + b2).shape)

## 4. 矩阵乘法 — `@` 是首选

**记忆口诀**：`(M, K) @ (K, N) = (M, N)`。**内侧 K 必须相等**，外侧 M、N 就是结果 shape。

**`np.matmul` / `@` vs `*`**：前者矩阵乘法；后者**逐元素**乘法。这俩搞混是新手第一坑。

In [ ]:
A = np.random.randn(3, 4)
B = np.random.randn(4, 2)

C = A @ B                    # 矩阵乘法：(3,4) @ (4,2) = (3,2)
print('A @ B  ->', C.shape)

# 同 shape 才能逐元素乘
D = np.random.randn(3, 4)
print('A * D  ->', (A * D).shape, '（逐元素，shape 不变）')

try:
    A * B            # shape (3,4) * (4,2) —— broadcasting 救不了，报错
except ValueError as e:
    print('A * B 翻车:', e)

In [ ]:
# 高维矩阵乘 —— 前置维度被当 batch，后两维做矩阵乘
B_dim, H, L, D = 2, 4, 8, 16    # batch, heads, seq_len, head_dim —— 注意力里常见 shape
Q = np.random.randn(B_dim, H, L, D)
K = np.random.randn(B_dim, H, L, D)

# K.transpose(0,1,3,2)：交换最后两维，变成 (B, H, D, L)
K_T = K.transpose(0, 1, 3, 2)
scores = Q @ K_T                # (B,H,L,D) @ (B,H,D,L) = (B,H,L,L)
print('attention scores shape:', scores.shape)

## 5. 常用归约 — `sum / mean / argmax / norm`

**关键参数**：`axis=` 指定沿哪一维归约，`keepdims=` 决定要不要保留那一维（保留 = 1）。

**LLM 场景**：softmax 沿最后一维归一、loss 沿 batch 求 mean、top-k 选 token —— 都靠它们。

In [ ]:
X = np.arange(12).reshape(3, 4).astype(float)
print(X, '\nshape:', X.shape)

print('\nsum 全部     :', X.sum())
print('sum axis=0    :', X.sum(axis=0), '<- 沿行方向，结果 shape (4,)')
print('sum axis=1    :', X.sum(axis=1), '<- 沿列方向，结果 shape (3,)')
print('sum axis=1, keepdims=True ->', X.sum(axis=1, keepdims=True).shape)
print('argmax axis=1 :', X.argmax(axis=1), '<- 每行最大元素的列号')

In [ ]:
# 手写 softmax —— 后面所有 attention/分类输出都用它
def softmax(x, axis=-1):
    # 减最大值是数值稳定技巧：防止 exp 上溢
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

logits = np.array([1.0, 2.0, 3.0])
p = softmax(logits)
print('logits:', logits)
print('softmax:', p)
print('和应为 1:', p.sum())

# 批量版本：(B, V)
batch_logits = np.array([[1.0, 2.0, 3.0],
                          [0.0, 0.0, 0.0]])
print('\nbatch softmax:\n', softmax(batch_logits, axis=-1))

## 6. 性能：为什么 NumPy 比 Python 循环快 100×

NumPy 把循环下沉到 C 实现 + 内存连续 + SIMD 向量化。**写深度学习代码的第一守则：避开 Python for-loop，全部向量化。**

In [ ]:
import time
n = 1_000_000
xs = np.random.randn(n)
ys = np.random.randn(n)

# 反例：Python 循环求点积
t0 = time.perf_counter()
s = 0.0
for i in range(n):
    s += xs[i] * ys[i]
py_time = time.perf_counter() - t0

# 正例：NumPy 一行
t0 = time.perf_counter()
s2 = (xs * ys).sum()
np_time = time.perf_counter() - t0

print(f'Python loop : {py_time*1000:.1f} ms')
print(f'NumPy vec  : {np_time*1000:.3f} ms')
print(f'加速倍数   : {py_time / np_time:.0f}x')
assert abs(s - s2) < 1e-6

## 深入思考

1. **`reshape` 和 `view` 是同一回事吗？**
   - NumPy 里 `reshape` 一般返回 view（共享内存）；shape 不兼容时会 copy。改 view 会影响原数组 —— **这是个坑**。
2. **为什么 `axis=0` 是「沿行方向」？**
   - 因为「沿 axis=k」意思是「把第 k 维收缩掉」。axis=0 收缩第 0 维（行数），结果 shape 是 `(cols,)`。换言之：**结果 shape 里那一维消失**。
3. **`X.T` 和 `X.transpose(...)` 区别？**
   - `.T` 是「翻转所有维」，2D 就是转置；高维用 `.transpose(0, 2, 1)` 这种显式形式更清楚。
4. **手写 softmax 为什么要减 max？**
   - `exp(1000)` 直接溢出成 `inf`。减 max 后最大值变 0，`exp(0)=1`，结果数学上不变（分子分母同除一个常数）。

改一改：把上一个 cell 的 `n` 改成 `10_000_000`，看加速倍数有没有变化。

## 自检 ✅

- [ ] 给一个 shape `(B, H, L, D)`，能立刻说出每一维是什么（batch / heads / seq_len / head_dim）。
- [ ] 写出 `(3, 4) @ (?, ?) = (3, 5)` 中 `?` 是多少。
- [ ] 闭眼写出 softmax 函数（含数值稳定）。
- [ ] 看到 `axis=1, keepdims=True`，能立刻说出结果 shape 的变化。
- [ ] 解释「`A * B` 和 `A @ B` 的区别」。

## 下一步

→ [`03_git_practice.ipynb`](03_git_practice.ipynb)